# Notebook 8 – Text Feature Engineering


## Learning objectives
- Explain how raw text becomes numeric features for machine learning.
- Extract text statistics such as length, word count, digits, punctuation, and keyword counts.
- Build Bag of Words, TF-IDF, and N-gram representations.
- Understand vocabulary, sparsity, preprocessing, and train/test leakage.

## 1. Why text feature engineering?
Machine-learning algorithms generally require numeric inputs. Raw sentences such as `"Payment was delayed"` must therefore be transformed into numbers.

Text feature engineering can produce:
1. **Statistical features** – word count, character count, digit count.
2. **Keyword features** – count or presence of domain terms.
3. **Vector features** – Bag of Words and TF-IDF.
4. **Context features** – N-grams such as `"payment delayed"`.

The right representation depends on the task, dataset size, language, and model.

In [ ]:
import re
import pandas as pd

df = pd.DataFrame({
    "message": [
        "Payment was delayed for 3 days!!!",
        "Customer requested REFUND immediately.",
        "Claim approved after document verification.",
        "URGENT: Policy number 45821 is missing.",
        "Customer is happy with the service."
    ],
    "category": ["complaint", "refund", "claim", "document", "positive"]
})
df

## 2. Text length and character count
`len(text)` counts characters, including spaces and punctuation. You may also calculate length after removing whitespace.

In [ ]:
df["character_count"] = df["message"].str.len()
df["characters_without_spaces"] = df["message"].str.replace(r"\\s+", "", regex=True).str.len()

df[["message", "character_count", "characters_without_spaces"]]

## 3. Word count and sentence count
A simple whitespace split is useful for demonstrations, but tokenization rules vary for contractions, hyphenated words, abbreviations, and languages without whitespace-separated words.

The sentence-count function below uses punctuation as a simple approximation, not a full linguistic sentence parser.

In [ ]:
df["word_count"] = df["message"].str.split().str.len()

def count_sentences(text):
    parts = re.split(r"[.!?]+", str(text).strip())
    return sum(bool(part.strip()) for part in parts)

df["sentence_count"] = df["message"].apply(count_sentences)
df[["message", "word_count", "sentence_count"]]

## 4. Digits, special characters, uppercase and lowercase counts
These can be useful in tasks such as fraud detection, document validation, spam detection, or support-ticket classification.

Define “special character” carefully. Here it means a character that is neither a letter, digit, nor whitespace.

In [ ]:
df["digit_count"] = df["message"].str.count(r"\\d")
df["special_character_count"] = df["message"].apply(
    lambda text: sum(not ch.isalnum() and not ch.isspace() for ch in text)
)
df["uppercase_count"] = df["message"].apply(
    lambda text: sum(ch.isupper() for ch in text)
)
df["lowercase_count"] = df["message"].apply(
    lambda text: sum(ch.islower() for ch in text)
)

df[["message", "digit_count", "special_character_count",
    "uppercase_count", "lowercase_count"]]

## 5. Keyword count
Keyword features count occurrences of terms associated with a domain. For an insurance-claims workflow, terms such as `claim`, `refund`, `urgent`, and `missing` may be useful.

Keyword rules are domain-specific. Consider case normalization, word boundaries, spelling variants, synonyms, and negation (for example, “not urgent”).

In [ ]:
keywords = ["payment", "refund", "claim", "urgent", "missing"]

for word in keywords:
    df[f"keyword_{word}"] = df["message"].str.lower().apply(
        lambda text: len(re.findall(rf"\\b{re.escape(word)}\\b", text))
    )

df[["message"] + [f"keyword_{word}" for word in keywords]]

## 6. Bag of Words (BoW)
Bag of Words converts text into a vector of word counts. Each vocabulary term becomes a column; each document becomes a row.

It ignores word order. For example, `"not good"` and `"good not"` can have the same unigram counts even though their meanings differ.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

texts = df["message"].tolist()
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(texts)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_vectorizer.get_feature_names_out()
)
bow_df

**Expected output:** A document-term matrix. A value of `1` means the word appeared once in that message; `0` means it did not appear. The exact columns are the learned vocabulary.

## 7. TF-IDF
TF-IDF means **Term Frequency–Inverse Document Frequency**. It gives more weight to terms that are frequent in a document but less common across the full collection.

A common conceptual form is:

\[
TFIDF(t,d) = TF(t,d) \times IDF(t)
\]

TF-IDF often reduces the influence of very common words and highlights terms that distinguish documents. It does not understand meaning in the human sense.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)
tfidf_df.round(3)

**Expected output:** A matrix of decimal weights. The values are generally not raw counts; they are TF-IDF weights calculated from this small corpus.

## 8. N-Grams
An N-gram is a contiguous sequence of N tokens:
- Unigram (1-gram): `payment`
- Bigram (2-gram): `payment delayed`
- Trigram (3-gram): `payment was delayed`

N-grams preserve some local word order and can capture phrases. Increasing N can create a much larger vocabulary and sparse matrices.

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_matrix = bigram_vectorizer.fit_transform(texts)

bigram_features = bigram_vectorizer.get_feature_names_out()
print("Number of unigram + bigram features:", len(bigram_features))
print("Sample features:", bigram_features[:20].tolist())

## 9. A practical text-feature table
Combine handcrafted numeric features with vectorized text features when there is a reason to do so. Avoid blindly adding every possible feature.

In [ ]:
text_feature_columns = [
    "character_count", "characters_without_spaces", "word_count",
    "sentence_count", "digit_count", "special_character_count",
    "uppercase_count", "lowercase_count"
]
df[["message"] + text_feature_columns].head()

## 10. Train/test leakage: a critical rule
Do not fit `CountVectorizer` or `TfidfVectorizer` on the entire dataset before splitting when evaluating a model. The vocabulary and IDF statistics should be learned from training text only.

Use a Pipeline so the vectorizer is fitted inside each training fold:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["category"],
    test_size=0.4, random_state=42, stratify=df["category"]
)

text_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000))
])

text_model.fit(X_train, y_train)
print("Test accuracy:", text_model.score(X_test, y_test))

**Note:** This tiny dataset is only for demonstrating the workflow. With five rows and five classes, the test metric is not statistically meaningful. Real text classification needs substantially more labelled examples and a suitable evaluation design.

## 11. Advantages and limitations

| Method | Advantages | Limitations |
|---|---|---|
| Text statistics | Simple, fast, interpretable | Does not capture semantic meaning |
| Keyword counts | Easy to explain; domain-focused | Depends on chosen vocabulary; misses synonyms |
| Bag of Words | Simple and effective baseline | Ignores word order; sparse; vocabulary-dependent |
| TF-IDF | Downweights common terms | Still mostly ignores deeper context and meaning |
| N-grams | Captures short phrases and local order | Can greatly increase feature count and sparsity |

